In [1]:
# ---------------------------------------------------
# Avalia o modelo fine-tunado do Pato Donald
# Métricas: BLEU, ROUGE-L, Donald-Score e Perplexity
# ---------------------------------------------------

import os, json, math, re, random, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import evaluate
from torch.nn import CrossEntropyLoss
from tqdm import tqdm

In [2]:
# ---------------------------------------------------
# Caminhos
# ---------------------------------------------------
BASE_MODEL = "google/gemma-3-4b-it"
LORA_PATH = "/mnt/data/jupyter-env/donaldDuck/donald_lora_out"
TEST_PATH = "/mnt/data/jupyter-env/donaldDuck/src/donald_synthetic_data/splits/test.jsonl"

In [3]:
# ---------------------------------------------------
# Carrega modelo e tokenizer
# ---------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (model): Gemma3Model(
        (vision_tower): SiglipVisionModel(
          (vision_model): SiglipVisionTransformer(
            (embeddings): SiglipVisionEmbeddings(
              (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
              (position_embedding): Embedding(4096, 1152)
            )
            (encoder): SiglipEncoder(
              (layers): ModuleList(
                (0-26): 27 x SiglipEncoderLayer(
                  (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                  (self_attn): SiglipAttention(
                    (k_proj): lora.Linear(
                      (base_layer): Linear(in_features=1152, out_features=1152, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.05, inplace=False)
                      )
                      (lor

In [4]:
# ---------------------------------------------------
# Carrega dados de teste
# ---------------------------------------------------
with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = [json.loads(l) for l in f if l.strip()]
print(f"{len(test_data)} exemplos carregados.")

# Subamostra para avaliação (100 exemplos)
subset = random.sample(test_data, min(10, len(test_data)))

111 exemplos carregados.


In [5]:
# ---------------------------------------------------
# Gera previsões
# ---------------------------------------------------
print("Gerando previsões do modelo...")
predictions, references = [], []
device = next(model.parameters()).device # Obter o device atual ('cuda')
max_len = 80 # Reuso da variável max_new_tokens
random.seed(42) # Para reprodutibilidade das amostras de geração (se usar do_sample)

# Parâmetros de geração para evitar o 'running off the rails' e dar estilo:
generation_args = {
    "max_new_tokens": max_len,
    "do_sample": True, # Ajuda a gerar respostas mais criativas e menos literais
    "temperature": 0.7,
    "top_k": 50,
    "top_p": 0.95,
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token_id": tokenizer.eos_token_id # Gemma usa EOS como PAD
}

for ex in tqdm(subset):
    # 1. Cria o PROMPT DE INFERÊNCIA (exatamente como o prompt de treinamento, mas SEM a resposta)
    prompt_infer = f"Usuário: {ex['user']}\nDonald:" 
    inputs = tokenizer(prompt_infer, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(**inputs, **generation_args)
    
    # 2. Decodificação e Pós-processamento (para extrair APENAS a resposta)
    full_output_text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Procura pela marca 'Donald:' e extrai tudo após ela.
    if "Donald:" in full_output_text:
        # Pega a parte APÓS "Donald:" e remove o próprio prompt
        pred = full_output_text.split("Donald:")[-1].strip()
        # Corta qualquer texto subsequente que se pareça com o próximo turno (após um '\n')
        pred = pred.split("\nUsuário:")[0].split("\n")[0].strip()
    else:
        # Se falhar, usa a geração completa (menos o prompt inicial)
        pred = full_output_text.replace(prompt_infer, "").strip()

    predictions.append(pred)
    references.append([ex["assistant"]]) # Lista de referências (necessária para BLEU)

print("Previsões geradas.")

Gerando previsões do modelo...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [01:09<00:00,  6.91s/it]

Previsões geradas.


In [6]:
# ---------------------------------------------------
# BLEU Score
# ---------------------------------------------------
print("Calculando BLEU...")
smoother = SmoothingFunction().method1
bleu = corpus_bleu(references, predictions, smoothing_function=smoother)
bleu

Calculando BLEU...


0.17206451279774987

In [7]:
# ---------------------------------------------------
# ROUGE-L
# ---------------------------------------------------
print("Calculando ROUGE-L...")
rouge = evaluate.load("rouge")
rouge_res = rouge.compute(predictions=predictions, references=[r[0] for r in references])
rouge_res

Calculando ROUGE-L...


{'rouge1': np.float64(0.21337572660801457),
 'rouge2': np.float64(0.07307386331425526),
 'rougeL': np.float64(0.198401363902692),
 'rougeLsum': np.float64(0.1982221516392541)}

In [10]:
# ---------------------------------------------------
# Perplexity (PPL)
# ---------------------------------------------------

import torch
import math
from torch.nn import CrossEntropyLoss
# Supondo que 'model' e 'tokenizer' já foram carregados e 'model.eval()' foi chamado.

# Função auxiliar para criar apenas o prompt, para fins de cálculo de comprimento de máscara.
def format_prompt_only(example):
    """Cria a string do prompt de inferência."""
    return f"Usuário: {example['user']}\nDonald:"

# Função auxiliar para criar o diálogo completo.
def format_example(example):
    """Cria a string completa do diálogo para tokenização."""
    return f"Usuário: {example['user']}\nDonald:{example['assistant']}"


print("Calculando Perplexity com Loss Masking...")
device = next(model.parameters()).device # Garante que o cálculo use a GPU
loss_fn = CrossEntropyLoss(ignore_index=-100, reduction='sum') # O ignore_index padrão do PyTorch é -100.
losses = []
total_tokens = 0
max_seq_length = 512 # Usando o mesmo valor de treinamento

model.eval() # Coloca o modelo em modo de avaliação

for ex in subset:  
    # 1. Preparação da Sequência
    full_dialogue = format_example(ex)
    prompt_only = format_prompt_only(ex)
    
    # 2. Tokenização da Sequência Completa
    inputs = tokenizer(
        full_dialogue, 
        return_tensors="pt", 
        truncation=True, 
        max_length=max_seq_length,
        padding=False # Não precisamos de padding para PPL aqui se estivermos em loop
    ).to(device)

    # 3. Determinação do Comprimento do Prompt
    # Tokeniza o prompt para saber onde começar o Loss
    prompt_tokens = tokenizer(
        prompt_only, 
        truncation=False, 
        add_special_tokens=False # CRUCIAL: Deve ser consistente com como o tokenizador trata a string
    )["input_ids"]
    prompt_len = len(prompt_tokens)
    
    # Se a tokenização for diferente, o `prompt_len` pode ser zero, 
    # ou maior que o comprimento total. Verificação de segurança:
    if prompt_len >= inputs["input_ids"].size(1):
         # Ignorar este exemplo se o prompt for maior ou igual à sequência tokenizada (rara)
         print(f"Aviso: Prompt muito longo ou vazio para o exemplo {ex.get('id', '')}. Ignorado.")
         continue

    # 4. Criação e Aplicação da Máscara (Labels)
    labels = inputs["input_ids"].clone()
    # Define os tokens do prompt como -100 para ignorar o loss
    labels[:, :prompt_len] = -100 
    
    # 5. Cálculo do Loss
    with torch.no_grad():
        # A função model() calcula o loss automaticamente ao receber o 'labels'
        outputs = model(**inputs, labels=labels)
        
        # O outputs.loss é a média de todos os tokens não-mascarados.
        # Precisamos do loss total e do número de tokens para a PPL.
        
        # Para garantir o cálculo exato do PPL (loss total sobre tokens válidos):
        
        # Remove o primeiro token (o Loss é sempre calculado do segundo token em diante)
        shift_logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        
        # Achatando e aplicando máscara (apenas tokens válidos)
        loss_logits = shift_logits.view(-1, shift_logits.size(-1))
        loss_labels = shift_labels.view(-1)
        
        # Identifica os tokens que NÃO foram mascarados (-100)
        valid_labels = loss_labels != -100
        
        # Calcula o loss TOTAL (soma) apenas para os tokens válidos
        token_loss = loss_fn(loss_logits[valid_labels], loss_labels[valid_labels])
        
        losses.append(token_loss.item())
        total_tokens += valid_labels.sum().item() # Conta os tokens que contribuíram para o loss

# 6. Cálculo da Perplexity Final
if total_tokens > 0:
    total_loss = sum(losses)
    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss)
else:
    ppl = float('inf')

print(f"\n--- Resultado da Perplexity ---")
print(f"Loss Total (mascarado): {total_loss:.4f}")
print(f"Tokens Válidos Contribuindo: {total_tokens}")
print(f"Perplexity Final: {ppl:.2f}")

Calculando Perplexity com Loss Masking...

--- Resultado da Perplexity ---
Loss Total (mascarado): 390.8569
Tokens Válidos Contribuindo: 234
Perplexity Final: 5.31


In [9]:
# ---------------------------------------------------
# Similaridade de Cosseno (Semantic Similarity)
# ---------------------------------------------------
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Calculando Similaridade de Cosseno (Semantic Similarity)...")

try:
    embedding_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2', device='cuda')
except Exception as e:
    print(f"Erro ao carregar SentenceTransformer. Tentando CPU: {e}")
    embedding_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2', device='cpu')

# 2. Preparar Dados
# Pegamos as previsões ('predictions') e as referências ('references')
# Note que 'references' é uma lista de listas, precisamos nivelá-la.
references_flat = [r[0] for r in references]

# 3. Gerar Embeddings
# O modelo gera um vetor (embedding) para cada frase.
embeddings_pred = embedding_model.encode(predictions, convert_to_tensor=True)
embeddings_ref = embedding_model.encode(references_flat, convert_to_tensor=True)

# 4. Calcular a Similaridade de Cosseno
# O cosine_similarity do scikit-learn espera arrays numpy/tensors, 
# mas o cálculo manual no tensor também é simples.

# Converter para Numpy para usar o scikit-learn (simples)
embeddings_pred_np = embeddings_pred.cpu().numpy()
embeddings_ref_np = embeddings_ref.cpu().numpy()

# A similaridade de cosseno é calculada par a par (predição[i] vs referência[i])
cosine_scores = np.diag(cosine_similarity(embeddings_pred_np, embeddings_ref_np))

# 5. Calcular a Média
avg_cosine_similarity = np.mean(cosine_scores)

print(f"Similaridade de Cosseno Média (por frase): {avg_cosine_similarity:.4f}")

# ---------------------------------------------------
# Opcional: Saída para análise
# ---------------------------------------------------
print("\nPrimeiros 5 Exemplos (Similaridade de Cosseno):")
for i in range(5):
    print(f"Score {i+1}: {cosine_scores[i]:.4f}")
    print(f"Pred: {predictions[i]}")
    print(f"Ref: {references_flat[i]}\n")

Calculando Similaridade de Cosseno (Semantic Similarity)...
Similaridade de Cosseno Média (por frase): 0.5506

Primeiros 5 Exemplos (Similaridade de Cosseno):
Score 1: 0.8065
Pred: Não! Hmpf! As pessoas parecem muito reais... e irritantes! Grrr! Bah! Quá-quá! É estranho! Hmpf! Quá-quá! Não, não, não! Grrr! Quá-quá! Hmpf! Não gosto! Grrr! Hmpf! Quá-quá! Hmpf
Ref: Assustador! Hmpf! Parecem gente de verdade. Grrr! Bah!

Score 2: 0.5749
Pred: Inteligência artificial? Mais uma coisa para me confundir! Hmpf! Grrr! Quá-quá! Não! Bah! Quá-quá! E muito barulhenta! Grrr! Bah! Hmpf! Quá-quá! Grrr! Sim, sim, e sim! Bah! Quá-quá! Grrr! Não!
Ref: IA? Mais uma coisa para me confundir! Quá-quá! Bah!

Score 3: 0.4182
Pred: Lido com elas! Bah! Quá-quá! Grrr! Sempre muito alto! Hmpf! Simplesmente insuportáveis! Bah! E o resultado é sempre o mesmo. Grrr! Quá-quá! Bah! E eu ainda sou o culpado. Hmpf! Tsk, tsk! Que vida! Quá-quá!
Ref: Expectativas? Bah! Eu não tenho nenhuma! Grrr! Assim não me decepciono! Q